# VL10 – Encoder–Decoder Transformers with BART and T5

In this lab we focus on **encoder–decoder Transformer models**, using **BART** and **T5** as concrete examples. These models are the backbone of many modern NLP applications such as summarisation, translation, and question answering.

Rather than building models from scratch, we work with **pretrained models from Hugging Face** and study how their design and pretraining objectives shape what they can do.

In this lab we will:

1. Load pretrained **BART and T5 models** and inspect their **tokenization and forward-pass outputs**.
2. Understand the **encoder–decoder architecture** and how it differs from encoder-only and decoder-only Transformers.
3. Explore the **pretraining objectives** of BART (denoising autoencoding) and T5 (span corruption).
4. Compare how these objectives affect what can (and cannot) be inspected or probed.
5. Fine-tune **BART and T5 on a summarisation task** using the CNN/DailyMail dataset.
6. Generate and qualitatively evaluate summaries produced by fine-tuned models.

## 1. Imports and device setup

We import:

- `transformers` for pretrained BERT models and tokenizer.
- `datasets` for a small IMDB sample.
- `torch` for tensors and optimization.

We also detect whether a GPU is available.

In [ ]:
import torch
import numpy as np

from datasets import load_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## 2. Loading pretrained models

There are multiple encoder–decoder Transformer models available on HuggingFace.
They differ in architecture, size, speed, and task formulation.

In this lab we focus on the smalles BART and T5

Models we will use
- **facebook/bart-base** ~554 MB (official version)
- **lucadiliello/bart-small** ~240 MB (community version)
- **trl-internal-testing/tiny-BartModel** 
- **google-t5/t5-small** ~240 MB (official version)
- **google/t5-efficient-tiny** ~64 MB (official version)

During class we use smaller base variants to keep training and inference
manageable on limited hardware.


### Downloading models on systems **without internet access in Jupyter**

Our environment does not allow internet downloads inside notebooks.  
We therefore download models **in the terminal** using a helper script:

```bash
$ python scripts/download_models.py "lucadiliello/bart-small" models/lucadiliello_bart-small
```
```bash
$ python scripts/download_models.py "google-t5/t5-small" models/google-t5_t5-small
```

This saves the entire model locally under models/

Once downloaded, we can load it offline from Jupyter by pointing to the local directory.


Now that we have chosen a model, we should download two components:

- `AutoTokenizer`: loading the **SentencePiece** subword tokenization.
- `AutoModel`: loading the **encoder-decoder** models.

These automatically choose the correct model architecture from the local folder.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

BART_MODEL = "../../models/lucadiliello_bart-small"
T5_MODEL = "../../models/google-t5_t5-small"

# BART
tokenizer_bart = AutoTokenizer.from_pretrained(BART_MODEL, local_files_only=True)
model_bart = AutoModelForSeq2SeqLM.from_pretrained(BART_MODEL, local_files_only=True).to(device)

# T5
tokenizer_t5 = AutoTokenizer.from_pretrained(T5_MODEL, local_files_only=True)
model_t5 = AutoModelForSeq2SeqLM.from_pretrained(T5_MODEL, local_files_only=True).to(device)

## 3. BART: Encoder–Decoder Transformers in Practice

BART is an **encoder–decoder Transformer** that combines bidirectional encoding with autoregressive generation. It is pretrained as a **denoising autoencoder**, learning to reconstruct clean text from corrupted input.

In this section, we briefly revisit BART from a practical perspective:
we inspect its tokenization, forward pass, and pretrained behavior, focusing on how the encoder and decoder interact in real applications.

### 3.1 Encoding Input Text

Input text is first processed using the **BART tokenizer**, which applies a subword tokenization (BPE - Byte Pair Encoding) scheme and adds model-specific special tokens.

The tokenizer:
- converts text into **token IDs**
- creates an **attention mask** indicating which tokens belong to the input and which are padding

In the tokenized output, the symbol **`Ġ`** marks the beginning of a token that follows a whitespace character. This is a feature of BART’s tokenizer and helps the model represent word boundaries.

In [ ]:
text = "Thank you for inviting me to your <mask> next week."

encoded = tokenizer_bart(text, return_tensors="pt").to(device)

# Let's extract the token ids and then convert it back to tokens 
# to see what the encoding has done
token_ids = encoded["input_ids"][0].tolist()
print(tokenizer_bart.convert_ids_to_tokens(token_ids))
print("token_ids     :", encoded["input_ids"][0].tolist())
print("attention_mask:", encoded["attention_mask"][0].tolist())


### 3.2 Forward Pass

We now run a forward pass through BART to **inspect the shapes of the encoder and decoder representations**.

Unlike encoder-only models, BART produces:
- **encoder hidden states** representing the input text
- **decoder hidden states** representing the generated sequence

By enabling `output_hidden_states`, we can explicitly observe:
- the dimensionality of encoder outputs
- the dimensionality of decoder outputs
- which intermediate representations the model exposes


In [ ]:
# Move inputs to device
encoded = {k: v.to(device) for k, v in encoded.items()}

with torch.no_grad():
    outputs = model_bart(
        **encoded,
        output_hidden_states=True,
        return_dict=True
    )

# Encoder representations (B, T, C)
encoder_hidden = outputs.encoder_last_hidden_state
print("Encoder hidden shape:", encoder_hidden.shape)

# Decoder representations (last layer: -1)
decoder_hidden = outputs.decoder_hidden_states[-1]
print("Decoder hidden shape:", decoder_hidden.shape)

# Notice that it will count embeddings as a layer, so it is actually len - 1
print("Number of encoder hidden-state layers:", len(outputs.encoder_hidden_states) -1)
print("Number of decoder hidden-state layers:", len(outputs.decoder_hidden_states) -1)

outputs.keys()

### 3.3 Denoising Pretraining in BART
BART is pretrained as a **denoising autoencoder**, where:

- The input sentence is **corrupted** (e.g. tokens removed, masked, permuted).
- The encoder processes the **corrupted text**.
- The decoder learns to **reconstruct the original sentence**.
- Training uses standard **sequence-to-sequence cross-entropy loss**.

Here, we inspect BART’s pretrained objective by giving it a **noisy sentence**
and observing how it _reconstructs_ it by filling out the mask.

In [ ]:
def bart_denoise(corrupted_text, tokenizer, model, max_length=50, num_beans=4):
    """
    Given a corrupted sentence, let BART reconstruct the original text.
    This mirrors BART's denoising pretraining objective.
    """

    print(f"Corrupted input:")
    print(f"  {corrupted_text}")

    # tokenize the input, and send it to the device the model resides in
    inputs = tokenizer(corrupted_text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_length=max_length,
            num_beams=num_beans
        )

    # ids to string
    reconstructed = tokenizer.decode(
        generated_ids[0], skip_special_tokens=True  # [0] as we have a batch dimension
    )

    print("\nReconstructed output:")
    print(f"  {reconstructed}")
    print("---")

    return reconstructed

**What is num_beams?** 
Beam search keeps several candidate continuations alive instead of just one. At each step, keep the N (e.g., N=4) most promising partial outputs and choose the best complete one at the end.

In [ ]:
bart_denoise("Thank you <mask> <mask> me next week.", tokenizer_bart, model_bart)

# Does it learn grammar?
bart_denoise("The child was <mask> the show on the TV.", tokenizer_bart, model_bart)
bart_denoise("She has <mask> to school.", tokenizer_bart, model_bart)

### Reflection
- Try different masked inputs. What kind of capabilities do you see emerging?
- Can it somehow infer positive / negative connotations? Try getting the rating for a movie given a comment:
    - The movie was ..... The rating was `<mask>`
  

## 4. T5: Text-to-Text Transformers

T5 is an **encoder–decoder Transformer** that frames *all* NLP tasks in a unified **text-to-text** format:  both inputs and outputs are plain text.

Unlike BART, T5 makes the **pretraining task explicit in the input** by using special *sentinel tokens* to mark missing spans.  This design choice has important consequences for how the model is trained, inspected, and prompted.

In this section, we examine T5’s tokenization, forward pass, and span-based pretraining objective, and contrast its behavior with BART.

### 4.1 Encoding Input Text

What difference do you notice?

In [ ]:
text = "Thank you for <extra_id_0> next <extra_id_1>."

encoded = tokenizer_t5(text, return_tensors="pt").to(device)

# Let's extract the token ids and then convert it back to tokens 
# to see what the encoding has done
token_ids = encoded["input_ids"][0].tolist()
print(tokenizer_t5.convert_ids_to_tokens(token_ids))
print("token_ids     :", encoded["input_ids"][0].tolist())
print("attention_mask:", encoded["attention_mask"][0].tolist())


Compared to BART, T5’s tokenization reveals several important differences:

- **No start-of-sequence token (`<s>`)**  
  The beginning of the sequence is implicit. T5 relies on position and SentencePiece markers instead of an explicit SOS token.

- **SentencePiece tokenization**  
  Tokens often start with the symbol **`▁`**, which marks the beginning of a word (i.e. a preceding whitespace).  
  This replaces BART’s `Ġ` convention.

- **Sentinel tokens (`<extra_id_n>`)**  
  T5 introduces special tokens such as `<extra_id_0>`, `<extra_id_1>`, …  
  These are atomic tokens used to explicitly mark missing spans during pretraining.

- **Explicit end-of-sequence token (`</s>`)**  
  The sequence end is still marked explicitly, as it is required for generation.


### 4.2 Forward-pass
To inspect the **internal representations** of T5 (encoder and decoder hidden states),
we must run a *training-style* forward pass using `labels`.

This explicitly defines the decoder input via *teacher forcing* and allows the
model to return all encoder and decoder representations in a single forward
pass. And it gives us access to those representations.

In contrast, `model.generate()` performs autoregressive decoding internally and
does not expose decoder hidden states as a single tensor.

**Note:**  
This is an API design choice: for T5, the Hugging Face forward pass requires an
explicit decoder sequence, which we provide via `labels`.


In [ ]:
# Example input and target (T5 pretraining style)
input_text = "Thank you for <extra_id_0> next <extra_id_1>."
target_text = "<extra_id_0> inviting me <extra_id_1> week."

encoded_inputs = tokenizer_t5(input_text, return_tensors="pt").to(device)
encoded_labels = tokenizer_t5(target_text, return_tensors="pt").input_ids.to(device)

with torch.no_grad():
    outputs = model_t5(
        input_ids=encoded_inputs["input_ids"],
        attention_mask=encoded_inputs["attention_mask"],
        labels=encoded_labels,
        output_hidden_states=True,
        return_dict=True
    )

encoder_hidden = outputs.encoder_last_hidden_state
decoder_hidden = outputs.decoder_hidden_states[-1]

print("Encoder hidden shape:", encoder_hidden.shape)
print("Decoder hidden shape:", decoder_hidden.shape)

# Notice that it will count embeddings as a layer, so it is actually len - 1
print("Number of encoder hidden-state layers:", len(outputs.encoder_hidden_states) -1)
print("Number of decoder hidden-state layers:", len(outputs.decoder_hidden_states) -1)

print("Output keys:", outputs.keys())

### 4.3 Span Reconstruction with T5

Unlike BART, T5 was pretrained with an **explicit span corruption objective**.

- Missing spans are replaced by **sentinel tokens** (`<extra_id_0>`, `<extra_id_1>`, …)
- The encoder receives the corrupted text
- The decoder is trained to **generate the missing spans**, in order
- Generation is **forced**: the model must output something for each sentinel

In this section, we inspect T5’s pretrained task by letting it
**fill in the missing spans** of a sentence.


In [ ]:
def t5_fill_spans(corrupted_text, tokenizer, model, num_beams=4, max_length=30):
    """
    Given a sentence containing <extra_id_n> tokens, let T5 generate
    the missing spans. This directly reflects T5's pretraining objective.
    """

    if "<extra_id_" not in corrupted_text:
        raise ValueError("Input must contain at least one <extra_id_n> token.")
        
    model.eval()
    print("\nCorrupted input:")
    print(f"  {corrupted_text}")

    inputs = tokenizer(corrupted_text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            num_beams=num_beams,
            early_stopping = True,
            eos_token_id=tokenizer.eos_token_id,
            
            max_new_tokens=max_length,
            no_repeat_ngram_size=2,
            repetition_penalty=1.2,
        )

    generated_text = tokenizer.decode(
        generated_ids[0], skip_special_tokens=False
    )

    print("\nGenerated output:")
    print(f"  {generated_text}")

    return generated_text

**Important**:
T5 does not generate the reconstructed sentence.
It generates only the missing spans, each preceded by a sentinel token.
Any text beyond the expected number of spans should be ignored.

In [ ]:
# Examples
_ = t5_fill_spans("I would like to thank you for <extra_id_0> regarding <extra_id_1>.", tokenizer_t5, model_t5)

_ = t5_fill_spans(
    "The city council approved <extra_id_0> after months of debate about <extra_id_1>.",
    tokenizer_t5,
    model_t5
)

_ = t5_fill_spans(
    "The dog <extra_id_0> in the yard <extra_id_1>.",
    tokenizer_t5,
    model_t5
)

**Note**: 
When using a purely pretrained T5 model (e.g. t5-small), span filling on _short
toy examples_ may result in empty or trivial outputs. This is expected and
reflects the maximum-likelihood pretraining objective.

## 5. Fine-Tuning Encoder–Decoder Models

So far, we explored **pretrained behavior** of BART and T5.
We now move to **fine-tuning**, where a pretrained encoder–decoder model is adapted to a specific downstream task.

In this section, we fine-tune BART and T5 on a **summarisation task** using the CNN/DailyMail dataset.  
The goal is to understand how pretrained sequence-to-sequence models can be efficiently adapted to real-world applications with relatively little task-specific data.

### 5.1 Dataset: CNN / DailyMail
We use the CNN/DailyMail dataset for abstractive summarisation.

- Input: news article
- Target: human-written summary
- This dataset works equally well for BART and T5

To keep fine-tuning manageable in class, we will use a **small subset**.

Download dataset:

````bash
$ python scripts/download_dataset.py "cnn_dailymail" "3.0.0" data/cnn_dailymail
````

In [ ]:
from datasets import load_dataset, load_from_disk

# Load dataset (version 3.0.0 is standard)
# dataset = load_dataset("cnn_dailymail", "3.0.0")
dataset = load_from_disk("../../data/cnn_dailymail")

# Optional: subsample for speed
train_ds = dataset["train"].shuffle(seed=42).select(range(2000))
test_ds   = dataset["test"].shuffle(seed=42).select(range(500))
val_ds   = dataset["validation"].shuffle(seed=42).select(range(500))

print(train_ds[0].keys())
dataset

In [ ]:
train_ds.to_pandas()

### 5.2 Preparing inputs for summarisation
BART and T5 share the same training procedure.
The only difference is input formatting:

- BART: input is the raw document
- T5: input is prefixed with `summarize:`

In [ ]:
def preprocess_summarization(examples, tokenizer, model_type="bart", max_input_length=512,
    max_target_length=128,
):
    """
    Prepare inputs and targets for summarisation fine-tuning.
    Works for both BART and T5.
    """

    # Input text
    if model_type == "t5":
        inputs = ["summarize: " + doc for doc in examples["article"]]
    else:
        inputs = examples["article"]

    targets = examples["highlights"]

    model_inputs = tokenizer(
        inputs,
        truncation=True,
        padding="max_length",
        max_length=max_input_length,
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            truncation=True,
            padding="max_length",
            max_length=max_target_length,
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

Let's apply the tokenization at the corpus level

In [ ]:
# Choose model type: "bart" or "t5"

def tokenize_corpus(ds, tokenizer, preprocess_for_task=preprocess_summarization, 
                    model_type="bart"):
    tokenized_ds = ds.map(
        lambda x: preprocess_for_task(x, tokenizer, model_type=model_type),
        batched=True,
        remove_columns=ds.column_names,
    )
    tokenized_ds.set_format("torch")
    return tokenized_ds

ds_train_bart = tokenize_corpus(train_ds, tokenizer_bart, model_type="bart")
ds_val_bart = tokenize_corpus(val_ds, tokenizer_bart, model_type="bart")

ds_train_t5 = tokenize_corpus(train_ds, tokenizer_t5, model_type="t5")
ds_val_t5 = tokenize_corpus(val_ds, tokenizer_t5, model_type="t5")


In [ ]:
ds_train_t5

### 5.3 Fine-Tuning Loop

At this stage, the model architectures and training objectives are well defined.
Rather than implementing a custom training loop, we rely on **existing Hugging Face utilities** that encapsulate standard fine-tuning procedures for encoder–decoder models.

Specifically, we use:

- **`Seq2SeqTrainer`**: manages the training and evaluation loop
- **`Seq2SeqTrainingArguments`**: configures how training is run (epochs, batch size, learning rate, etc.)
- **`DataCollatorForSeq2Seq`**: handles batching, padding, and label shifting for sequence-to-sequence tasks

These components allow us to focus on **model behavior and task performance**, while abstracting away boilerplate training code.

Depending on the environment, some of these utilities might trigger some dependency issues with `accelerate`. If this happens, try installing or updating `accellerate` with:

````bash
$ pip install "accelerate>=0.26.0"
````

In [ ]:
import torch
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)

def setup_encoder_decoder(params):
    """
    Build a Trainer for encoder–decoder summarisation models (BART or T5).
    """

    model = params["model"]
    tokenizer = params["tokenizer"]

    # 1. Training configuration
    training_args = Seq2SeqTrainingArguments(
        output_dir=params.get("output_dir", "./results"),

        # NOTE: older HF versions use `eval_strategy`
        eval_strategy="epoch",

        learning_rate=params.get("learning_rate", 2e-5),
        per_device_train_batch_size=params.get("train_batch_size", 4),
        per_device_eval_batch_size=params.get("eval_batch_size", 4),
        gradient_accumulation_steps=params.get("grad_accum_steps", 2), # perform optimizer.step() every N batches
        num_train_epochs=params.get("epochs", 3),

        save_total_limit=1,  # how many checkpoints are kept on disk. (1 = most recent)
        logging_steps=4,     # how often training statistics are printed/logged.
        fp16=torch.cuda.is_available(), # mixed-precision training (instead of fp32)
        report_to="none",

        # seq2seq-specific
        predict_with_generate=True, # use mode.generate() to get full decoded sequences
        generation_max_length=params.get("gen_max_length", 128),
    )

    # 2. Data collator
    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
    )

    # 3. Trainer
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=params["train_dataset"],
        eval_dataset=params["eval_dataset"],
        tokenizer=tokenizer,
        data_collator=data_collator,
    )

    return trainer


### 5.4 Single inference step
The function `perform_task_one` runs a **single inference step** for an encoder–decoder model.

It takes a raw input text (e.g. a news article), prepares it according to the model
type, and uses `model.generate()` to produce an output sequence.

- For **T5**, a task prefix (e.g. `summarize:`) is prepended to the input text, as
  required by T5’s text-to-text design.
- For **BART**, the input text is passed directly, since the task is implicit.

The function handles tokenization, moves inputs to the correct device, performs
generation with beam search, and returns the decoded output text.

In [ ]:
import torch

def perform_task_one(
    model,
    tokenizer,
    article: str,
    model_type: str = "bart",   # "bart" or "t5"
    t5_task = None,
    max_input_length: int = 512,
    max_new_tokens: int = 80,
    num_beams: int = 4,
):
    model.eval()

    # T5 expects a task prefix
    if model_type == "t5":
        article_in = t5_task + ": " + article
    else: # bart
        article_in = article

    inputs = tokenizer(
        article_in,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length,
    ).to(model.device)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
        )

    return tokenizer.decode(out_ids[0], skip_special_tokens=True)


### 5.5 Fine-tunning BART

In [ ]:
params = {
    "model": model_bart,
    "tokenizer": tokenizer_bart,
    "train_dataset": ds_train_bart,
    "eval_dataset": ds_val_bart,
    "epochs": 5,
    "output_dir": "./bart_summarization"    
}

trainer_bart = setup_encoder_decoder(params)
trainer_bart.train()

#### Let's now try the model:

In [ ]:
# Pick one validation example
i = 3
ex = dataset["test"][i]
article = ex["article"]
gold = ex["highlights"]

print("ARTICLE (truncated):\n", article[:1000], "...\n")
print("GOLD SUMMARY:\n", gold, "\n")

pred = perform_task_one(model_bart, tokenizer_bart, article, model_type="bart")
print("MODEL SUMMARY:\n", pred)

### 5.6 Fine-tunning T5

In [ ]:
params = {
    "model": model_t5,
    "tokenizer": tokenizer_t5,
    "train_dataset": ds_train_t5,
    "eval_dataset": ds_val_t5,
    "epochs": 5,
    "output_dir": "./t5_summarization",
}

trainer_t5 = setup_encoder_decoder(params)
trainer_t5.train()

#### Let's now try the model:

In [ ]:
# Validation example
# ex, article, gold -> defined above in BART

print("ARTICLE (truncated):\n", article[:1000], "...\n")
print("GOLD SUMMARY:\n", gold, "\n")

pred = perform_task_one(model_t5, tokenizer_t5, article, model_type="t5", t5_task="summarization")
print("T5 MODEL SUMMARY:\n", pred)

## Reflection

1. Compare T5 and BART on validation loss. Which one converges faster, and whith one reaches the lower validation loss?
2. Compare BART and T5 on the same 2–3 validation articles. Which model produces the more informative and fluent summaries?
3. For T5, remove the `summarize:` prefix  from `perform_task_one` and observe what changes. Why is this prefix important for T5?
4. Try increasing the amount of training data or the number of epochs. Does the validation loss improve, or does the model begin to overfit?

